# imports

In [ ]:
from pathlib import Path
from typing import Any
import json

from jaxtyping import Float
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
import umap

# muutils
import muutils.tensor_info
from muutils.dbg import dbg_tensor

# attention-motifs
from attention_motifs.features.analysis import (
	DistanceTensorResult,
)
from attention_motifs.features.plotting import (
	plot_embedding,
)
from attention_motifs.attnpedia.attnpedia import AttentionPedia

In [ ]:
# magic autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
# numbers of rows to display
pl.Config.set_tbl_rows(10)
# colored muutils tensor info
muutils.tensor_info.DEFAULT_SETTINGS["colored"] = True
# base path
PATH_BASE: Path = Path("../data/features/")

# load data

In [ ]:
DF_PCA: pl.DataFrame = pl.read_ndjson(PATH_BASE / "pca.jsonl")

# distances between heads

In [ ]:
HEAD_DISTS: DistanceTensorResult = DistanceTensorResult.build_distance_tensor(
	DF_PCA,
	feature_prefix="pc.",
)
dbg_tensor(HEAD_DISTS.distances)
dbg_tensor(HEAD_DISTS.mean_dists)
HEAD_DISTS.save_means(PATH_BASE / "head_dists.json")
print()

In [ ]:
def parse_cls(cls: str) -> tuple[str, int, int]:
	"""
	Split ``{model}:L{layer}:H{head}`` into (model, layer, head).
	"""
	model_part, layer_part, head_part = cls.split(":")
	layer = int(layer_part[1:])  # drop leading "L"
	head = int(head_part[1:])  # drop leading "H"
	return model_part, layer, head


# --------------------------- plot ---------------------------------- #
def plot_heatmap(
	self,
	figsize: tuple[int, int] | None = (10, 10),
	stripe_frac: float = 0.025,
	label_frac: float = 0.05,
	model_fontsize: int = 9,
	layer_fontsize: int = 7,
	grid_alpha_major: float = 0.4,
	grid_alpha_minor: float = 0.2,
	show: bool = True,
) -> plt.Figure:
	"""
	Visualise the distance matrix with model/layer encoding and grid.

	* Sorts heads by (model, layer, head).
	* Two colour stripes (top + left) encode model + depth.
	* Both *model names* and *first/last layer* numbers appear on
		**both** axes (model labels outside, layer labels inside stripes).
	* Subtle grid: major lines at model boundaries, minor lines at layers.

	Returns
	-------
	figure : matplotlib.figure.Figure
	"""
	# ----------------------- sorting ---------------------------------
	parsed = [parse_cls(c) for c in self.cls_values]
	sort_idx = sorted(range(len(self.cls_values)), key=lambda i: parsed[i])
	dists_sorted = self.mean_dists[np.ix_(sort_idx, sort_idx)]
	parsed_sorted = [parsed[i] for i in sort_idx]
	n = len(parsed_sorted)

	# ----------------------- colours ---------------------------------
	unique_models = sorted({p[0] for p in parsed_sorted})
	base_cmap = plt.get_cmap("tab20")
	model_to_rgb = {
		m: base_cmap(i % base_cmap.N)[:3] for i, m in enumerate(unique_models)
	}

	max_layer: dict[str, int] = {}
	for model, layer, _ in parsed_sorted:
		max_layer[model] = max(max_layer.get(model, -1), layer)

	rgba = []
	for model, layer, _ in parsed_sorted:
		base = np.asarray(model_to_rgb[model])
		scale = 0.3 + 0.7 * (layer / max_layer[model] if max_layer[model] else 0.0)
		rgb = base * scale + (1.0 - scale)
		rgba.append(tuple(rgb) + (1.0,))
	stripe_h = np.array(rgba).reshape(1, -1, 4)
	stripe_v = np.array(rgba).reshape(-1, 1, 4)

	# ----------------------- figure axes ----------------------------
	fig, ax_main = plt.subplots(figsize=figsize)
	im = ax_main.matshow(dists_sorted)
	ax_main.set_xticks([])
	ax_main.set_yticks([])
	ax_main.set_aspect("equal")

	# ----------------------- stripes --------------------------------
	ax_top = ax_main.inset_axes(
		[0, 1.0 + label_frac, 1, stripe_frac],
		transform=ax_main.transAxes,
		sharex=ax_main,
	)
	ax_top.imshow(stripe_h, aspect="auto", origin="lower", extent=[-0.5, n - 0.5, 0, 1])
	ax_top.set_axis_off()

	ax_left = ax_main.inset_axes(
		[-stripe_frac - label_frac, 0, stripe_frac, 1],
		transform=ax_main.transAxes,
		sharey=ax_main,
	)
	ax_left.imshow(
		stripe_v, aspect="auto", origin="upper", extent=[0, 1, -0.5, n - 0.5]
	)
	ax_left.set_axis_off()

	# ----------------------- model label axes -----------------------
	# top outer axis
	ax_top_model = ax_main.inset_axes(
		[0, 1.0 + label_frac + stripe_frac + 0.005, 1, label_frac],
		transform=ax_main.transAxes,
		sharex=ax_main,
	)
	ax_top_model.set_axis_off()

	# left outer axis
	ax_left_model = ax_main.inset_axes(
		[-label_frac - stripe_frac - 0.005, 0, label_frac, 1],
		transform=ax_main.transAxes,
		sharey=ax_main,
	)
	ax_left_model.set_axis_off()

	# ----------------------- spans / boundaries ---------------------
	model_spans: dict[str, tuple[int, int]] = {}
	for idx, (model, layer, _) in enumerate(parsed_sorted):
		if model not in model_spans:
			model_spans[model] = [idx, idx]
		model_spans[model][1] = idx

	# boundaries lists
	model_bounds = []
	layer_bounds = []

	last_model, last_layer = parsed_sorted[0][:2]
	for i in range(1, n):
		model, layer, _ = parsed_sorted[i]
		if model != last_model:
			model_bounds.append(i - 0.5)
		elif layer != last_layer:
			layer_bounds.append(i - 0.5)
		last_model, last_layer = model, layer

	# ----------------------- grid lines -----------------------------
	for b in model_bounds:
		ax_main.axvline(x=b, color="k", linewidth=1.2, alpha=grid_alpha_major, zorder=2)
		ax_main.axhline(y=b, color="k", linewidth=1.2, alpha=grid_alpha_major, zorder=2)
	for b in layer_bounds:
		ax_main.axvline(x=b, color="k", linewidth=0.6, alpha=grid_alpha_minor, zorder=2)
		ax_main.axhline(y=b, color="k", linewidth=0.6, alpha=grid_alpha_minor, zorder=2)

	# ----------------------- annotations ---------------------------
	# model names on top and left
	for model, (s, e) in model_spans.items():
		mid = (s + e) / 2
		# top horizontal
		ax_top_model.text(
			mid, 0.5, model, ha="center", va="center", fontsize=model_fontsize
		)
		# left vertical (rotated)
		ax_left_model.text(
			0.5,
			mid,
			model,
			ha="center",
			va="center",
			fontsize=model_fontsize,
			rotation=90,
		)

	# layer numbers (first and last) inside stripes (both axes)
	for model, (s, e) in model_spans.items():
		layer_first = parsed_sorted[s][1]
		layer_last = parsed_sorted[e][1]
		# top stripe
		ax_top.text(
			s,
			0.5,
			f"L{layer_first}",
			ha="center",
			va="center",
			fontsize=layer_fontsize,
			rotation=90,
		)
		if layer_last != layer_first:
			ax_top.text(
				e,
				0.5,
				f"L{layer_last}",
				ha="center",
				va="center",
				fontsize=layer_fontsize,
				rotation=90,
			)
		# left stripe
		ax_left.text(
			0.5, s, f"L{layer_first}", ha="center", va="center", fontsize=layer_fontsize
		)
		if layer_last != layer_first:
			ax_left.text(
				0.5,
				e,
				f"L{layer_last}",
				ha="center",
				va="center",
				fontsize=layer_fontsize,
			)

	# ---------------------- colour bar ------------------------------
	cbar = fig.colorbar(im, ax=ax_main, fraction=0.046, pad=0.04)
	cbar.ax.set_ylabel("Mean distance", rotation=270, labelpad=15)

	if show:
		plt.show()

	return fig


plot_heatmap(HEAD_DISTS, show=True)

In [ ]:
ATTENTIONPEDIA: AttentionPedia = AttentionPedia()

In [ ]:
HEAD_DISTS.get_closest_heads("gpt2-small:L5:H5", n_closest=10)
# 'gpt2-medium:L5:H8

# for k in HEAD_DISTS.cls_values:
# 	plt.plot(
# 		[
# 			x[1]
# 			for x in HEAD_DISTS.get_closest_heads(k, n_closest=50)[1:]
# 		],
# 		alpha=0.05,
# 		color="black",
# 	)

# plt.show()

In [ ]:
# HEAD_DISTS.plot_hists(n_samples=64)

In [ ]:
from attention_motifs.features.head_analysis import filter_umap_warns


filter_umap_warns()

n_neighbors_lst: list[int] = [2, 4, 8, 16]  # , 32, 64, 128, 256, 512]
fig, ax_hm = plt.subplots(
	nrows=1,
	ncols=len(n_neighbors_lst),
	figsize=(10 * len(n_neighbors_lst), 10),
)
for i, n_neighbors in enumerate(n_neighbors_lst):
	reducer: umap.UMAP = umap.UMAP(
		n_components=2,
		metric="precomputed",
		init="spectral",
		n_neighbors=n_neighbors,
		min_dist=0.1,
		random_state=0,
	)
	embedding: np.ndarray = reducer.fit_transform(HEAD_DISTS.mean_dists)

	# scatter plot
	attnpedia_head_to_type: dict[str, str] = AttentionPedia().head_to_type()
	plot_embedding(
		embedding,
		# labels=pl.Series([f.split(":")[0] for f in head_dists.cls_values]),
		labels=pl.Series(
			[attnpedia_head_to_type.get(f, "unknown") for f in HEAD_DISTS.cls_values]
		),
		dims=(0, 1),
		alpha={
			"unknown": 0.2,
			None: 0.9,
		},
		marker_size={
			"unknown": 10,
			None: 30,
		},
		title=f"UMAP embedding ({n_neighbors = })",
		ax=ax_hm[i],
	)
plt.show()

# plt.figure(figsize=(8, 6))
# plt.scatter(embedding[:, 0], embedding[:, 1], s=10)  # no explicit colours
# plt.xlabel("UMAP 0")
# plt.ylabel("UMAP 1")
# plt.title("UMAP embedding")
# plt.tight_layout()
# plt.show()

In [ ]:
def plot_hclust(
	dist_mat: Float[np.ndarray, "h h"],
	labels: list[str],
	*,
	method: str = "average",
	leaf_font_size: int = 6,
	truncate: int | None = None,
	show_heatmap: bool = True,
) -> tuple[np.ndarray, list[str], dict[str, Any]]:
	"""Hierarchical clustering dendrogram (plus optional heat-map).

	# Parameters
	 - `dist_mat : Float[np.ndarray, "h h"]`
	    Square distance matrix (symmetric, zero diagonal).
	 - `labels : list[str]`
	    Observation names, length `h`.
	 - `method : str`
	    Linkage criterion for `scipy.cluster.hierarchy.linkage`.
	 - `leaf_font_size : int`
	    Tick-label font size.
	 - `truncate : int | None`
	    If given, plot the dendrogram in *last-p* mode with `p = truncate`.
	 - `show_heatmap : bool`
	    Draw reordered distance matrix.
	 - `figsize : tuple[int, int] | None`
	    Figure size; if `None`, pick a reasonable default.

	# Returns
	 - `np.ndarray`
	    Linkage matrix `Z` (`h - 1`, 4).
	 - `dict[str, Any]`
	    Dot-list representation of the tree **as actually plotted**
	    (clusters are collapsed when `truncate` is used).
	"""
	h: int = dist_mat.shape[0]

	# ---------- sanity checks
	if dist_mat.shape != (h, h):
		raise ValueError("dist_mat must be square")
	if len(labels) != h:
		raise ValueError("labels length must match dist_mat")
	if not np.allclose(dist_mat, dist_mat.T):
		raise ValueError("dist_mat must be symmetric")
	if not np.all(np.diag(dist_mat) == 0):
		raise ValueError("dist_mat diagonal must be zero")

	# ---------- linkage
	condensed: Float[np.ndarray, "h*(h-1)//2"] = squareform(dist_mat, checks=False)
	Z: np.ndarray = linkage(condensed, method=method)

	# ---------- dendrogram
	_, ax_dend = plt.subplots(figsize=(8, 4))
	dend_kw: dict[str, object] = {
		"labels": labels,
		"leaf_rotation": 90,
		"leaf_font_size": leaf_font_size,
		"distance_sort": "ascending",
	}
	if truncate is not None:
		dend_kw.update({"truncate_mode": "lastp", "p": truncate})

	dendro: dict[str, Any] = dendrogram(Z, **dend_kw, ax=ax_dend)
	ax_dend.set_ylabel("Distance")
	ax_dend.set_title(f"Hierarchical clustering ({method} linkage)")
	# ax_dend.set_ylim(7, 27)
	plt.tight_layout()
	plt.show()

	# ---------- helpers
	def _members(node_id: int, *, n: int = h) -> list[int]:
		"""Return all original leaf indices contained in `node_id`."""
		if node_id < n:
			return [node_id]
		left: int = int(Z[node_id - n, 0])
		right: int = int(Z[node_id - n, 1])
		return _members(left, n=n) + _members(right, n=n)

	visible_ids: set[int] = set(dendro["leaves"])

	# ---------- heat-map
	order: list[int]
	if truncate is None:
		order = list(dendro["leaves"])
	else:
		order = [idx for node in dendro["leaves"] for idx in _members(node)]
	labels_ordered: list[str] = [labels[i] for i in order]
	if show_heatmap:
		reordered: Float[np.ndarray, "h h"] = dist_mat[np.ix_(order, order)]

		_, ax_hm = plt.subplots(figsize=(30, 30))
		im = ax_hm.imshow(reordered, aspect="auto")
		ax_hm.set_xticks(np.arange(h))
		ax_hm.set_yticks(np.arange(h))
		ax_hm.set_xticklabels(labels_ordered, rotation=90, fontsize=leaf_font_size)
		ax_hm.set_yticklabels(labels_ordered, fontsize=leaf_font_size)
		ax_hm.set_xlabel("Classes")
		ax_hm.set_ylabel("Classes")
		ax_hm.set_title("Distance matrix (reordered)")
		ax_hm.set_aspect("equal")
		plt.colorbar(im, ax=ax_hm)
		plt.tight_layout()
		plt.show()

	# ---------- dot-list
	def _dot_tree(node_id: int, path: str = "") -> dict[str, Any]:
		key: str = path.rstrip(".") or "root"
		if node_id in visible_ids:  # displayed leaf/cluster
			mem: list[int] = _members(node_id)
			return {key: labels[mem[0]] if len(mem) == 1 else [labels[m] for m in mem]}
		left, right = (int(x) for x in Z[node_id - h, :2])
		dot: dict[str, Any] = {}
		dot.update(_dot_tree(left, f"{path}0."))
		dot.update(_dot_tree(right, f"{path}1."))
		return dot

	tree_dict: dict[str, Any] = _dot_tree(2 * h - 2)

	return Z, labels_ordered, tree_dict


linkages, labels_sorted, tree = plot_hclust(
	HEAD_DISTS.mean_dists,
	labels=HEAD_DISTS.cls_values,
	truncate=100,
	# show_heatmap=False,
)

dbg_tensor(linkages)
leaf_lens: dict[str, int] = {k: len(v) for k, v in tree.items()}
plt.hist(
	leaf_lens.values(),
	bins=50,
)
plt.show()
print(
	json.dumps(
		leaf_lens,
		indent=2,
	)
)

In [ ]:
import itertools

tree_known_types: dict[str, set[str]] = {
	k: set(
		itertools.chain.from_iterable(
			[ATTENTIONPEDIA.head_to_types().get(h, ["unknown"]) for h in v]
		)
	)
	for k, v in tree.items()
}
tree_known_types

In [ ]:
known_types_to_leaf: dict[str, set[str]] = {
	t: set([head for head, types in tree_known_types.items() if t in types])
	for t in [*ATTENTIONPEDIA.type_to_heads().keys(), "unknown"]
}
known_types_to_leaf